# 22.07 - Video baseline experiment

**Notebook type:** Solution notebook with theory, completed exercises, smoke checks, and test cases.

**Daily output:** Video baseline from scratch.

Build the complete timed path: sample frames → run a shared CNN → average frame logits → compute a validation metric → create a submission-style CSV table.

## Core Ideas

A strong first video baseline reuses an image classifier. Sample a fixed number of frames uniformly, reshape `[B,T,C,H,W]` into `[B*T,C,H,W]`, classify every frame, restore `[B,T,K]`, and average logits across time.

Uniform sampling makes variable-length videos comparable. Averaging logits gives every sampled frame equal influence and keeps the pipeline vectorized. Validation and submission must use the same sampling, normalization, class-index mapping, and test-ID order.

This toy experiment uses deterministic in-memory clips so the notebook runs quickly without downloads. The two classes differ by whether the bright region is on the left or right.

## Prepared Video Data

The prepared tensors follow `[N,T,C,H,W]`, use `float32`, and contain integer class labels in `{0,1}`. Data preparation is provided and is not a learner TODO.

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_labels = torch.arange(24, dtype=torch.long) % 2
val_labels = torch.arange(8, dtype=torch.long) % 2
test_pattern = torch.arange(6, dtype=torch.long) % 2
train_videos = 0.05 * torch.randn(24, 8, 1, 8, 8)
val_videos = 0.05 * torch.randn(8, 8, 1, 8, 8)
test_videos = 0.05 * torch.randn(6, 8, 1, 8, 8)
for videos, labels in [(train_videos, train_labels), (val_videos, val_labels), (test_videos, test_pattern)]:
    for index, label in enumerate(labels.tolist()):
        if label == 0:
            videos[index, :, :, :, :4] += 1.0
        else:
            videos[index, :, :, :, 4:] += 1.0
test_ids = [f'video_{index:03d}' for index in range(len(test_videos))]
print(train_videos.shape, train_videos.dtype, train_labels.shape, DEVICE)

## Exercise 22-A: Uniform frame sampling

Select `n_frames` evenly spaced temporal positions, including the first and last frame when more than one frame is requested.

**Return structure — `sample_uniform_frames`:** accepts a `torch.Tensor` shaped `[B,T,C,H,W]` and returns a tensor shaped `[B,S,C,H,W]`, where `S=n_frames`. The return keeps the input dtype and device. If `S>T`, repeated temporal indices are allowed.

In [ ]:
def sample_uniform_frames(videos, n_frames):
    if videos.ndim != 5:
        raise ValueError('videos must have shape [B,T,C,H,W]')
    if n_frames < 1:
        raise ValueError('n_frames must be positive')
    indices = torch.linspace(0, videos.shape[1] - 1, n_frames, device=videos.device).round().long()
    return videos.index_select(1, indices)


# Smoke check: exercise the completed function immediately.
smoke_sample = sample_uniform_frames(train_videos[:2], 4)
print('sampled:', smoke_sample.shape, smoke_sample.dtype, smoke_sample.device)

## Exercise 22-B: Frame classifier

Create a tiny CNN that retains enough spatial layout to distinguish the bright-left and bright-right classes.

**Return structure — `TinyFrameCNN`:** constructing the class returns an `nn.Module` instance. Calling it with a floating-point tensor shaped `[M,C,H,W]` on the module device returns floating-point logits shaped `[M,K]` on that device, where `K=num_classes`.

In [ ]:
class TinyFrameCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((2, 2)),
        )
        self.classifier = nn.Linear(8 * 2 * 2, num_classes)

    def forward(self, frames):
        return self.classifier(self.features(frames).flatten(1))


# Smoke check: exercise the completed class immediately.
frame_model = TinyFrameCNN().to(DEVICE)
smoke_frame_logits = frame_model(train_videos[:2, 0].to(DEVICE))
print('frame logits:', smoke_frame_logits.shape, smoke_frame_logits.device)

## Exercise 22-C: Average frame logits

Vectorize frame inference and average across the restored time dimension.

**Return structure — `video_average_logits`:** accepts an `nn.Module`, a floating-point tensor `[B,T,C,H,W]` on the model device, and integer `n_frames`; returns floating-point video logits `[B,K]` on the same device.

In [ ]:
def video_average_logits(model, videos, n_frames=4):
    sampled = sample_uniform_frames(videos, n_frames)
    batch_size, sample_count, channels, height, width = sampled.shape
    frame_logits = model(sampled.reshape(batch_size * sample_count, channels, height, width))
    return frame_logits.reshape(batch_size, sample_count, -1).mean(dim=1)


# Smoke check: exercise the completed function immediately.
smoke_video_logits = video_average_logits(frame_model, train_videos[:3].to(DEVICE), 4)
print('video logits:', smoke_video_logits.shape, smoke_video_logits.device)

## Exercise 22-D: Train the baseline

Run one full-batch optimization step through the video-level average-logit pipeline.

**Return structure — `train_video_baseline_step`:** returns one Python `float` containing the non-negative cross-entropy loss for the completed optimizer step. Model parameters are updated as a side effect; the input tensors are not modified.

In [ ]:
def train_video_baseline_step(model, videos, labels, optimizer, n_frames=4):
    model.train()
    optimizer.zero_grad()
    logits = video_average_logits(model, videos, n_frames)
    loss = nn.functional.cross_entropy(logits, labels)
    loss.backward()
    optimizer.step()
    return float(loss.detach().cpu().item())


# Smoke check: train the model and show that the objective changes.
optimizer = torch.optim.Adam(frame_model.parameters(), lr=0.05)
loss_history = [
    train_video_baseline_step(frame_model, train_videos.to(DEVICE), train_labels.to(DEVICE), optimizer)
    for _ in range(12)
]
print('first/final loss:', loss_history[0], loss_history[-1])

## Exercise 22-E: Validation Macro-F1

Compute F1 independently for every class, then average class scores so minority classes matter equally.

**Return structure — `macro_f1_numpy`:** accepts two one-dimensional integer-like arrays of equal length and returns one Python `float` in `[0.0,1.0]`. Classes are the sorted union of values in targets and predictions; a zero-denominator class receives F1 `0.0`.

In [ ]:
def macro_f1_numpy(targets, predictions):
    targets = np.asarray(targets).reshape(-1)
    predictions = np.asarray(predictions).reshape(-1)
    if targets.shape != predictions.shape or targets.size == 0:
        raise ValueError('targets and predictions must be non-empty and equally shaped')
    class_scores = []
    for class_id in np.union1d(targets, predictions):
        true_positive = np.sum((targets == class_id) & (predictions == class_id))
        false_positive = np.sum((targets != class_id) & (predictions == class_id))
        false_negative = np.sum((targets == class_id) & (predictions != class_id))
        denominator = 2 * true_positive + false_positive + false_negative
        class_scores.append(0.0 if denominator == 0 else float(2 * true_positive / denominator))
    return float(np.mean(class_scores))


# Smoke check: run validation immediately.
frame_model.eval()
with torch.no_grad():
    val_logits = video_average_logits(frame_model, val_videos.to(DEVICE), 4)
val_predictions = val_logits.argmax(dim=1).cpu().numpy()
val_macro_f1 = macro_f1_numpy(val_labels.numpy(), val_predictions)
print('validation Macro-F1:', val_macro_f1)

## Exercise 22-F: Submission-style table

Preserve test ID order and map each model prediction to one integer label.

**Return structure — `make_submission`:** returns a `pandas.DataFrame` with exactly two columns in order: `video_id` (string-like values copied from `test_ids`) and `label` (`int64`). It has exactly one row per test ID, preserves input order, and contains no missing values.

In [ ]:
def make_submission(test_ids, predictions):
    predictions = np.asarray(predictions, dtype=np.int64).reshape(-1)
    if len(test_ids) != len(predictions):
        raise ValueError('test_ids and predictions must have equal length')
    return pd.DataFrame({'video_id': list(test_ids), 'label': predictions})


# Smoke check: create and inspect the submission table immediately.
frame_model.eval()
with torch.no_grad():
    test_predictions = video_average_logits(frame_model, test_videos.to(DEVICE), 4).argmax(1).cpu().numpy()
submission = make_submission(test_ids, test_predictions)
print(submission)

## Test Cases

A correct solution prints `Day 22 tests passed`.

**Return structure — `run_day22_tests`:** returns `None`. Success is communicated by completing all assertions and printing exactly `Day 22 tests passed`; a failed requirement raises `AssertionError`.

In [ ]:
def run_day22_tests():
    sampled = sample_uniform_frames(train_videos[:2], 4)
    assert sampled.shape == (2, 4, 1, 8, 8)
    assert sampled.dtype == train_videos.dtype and sampled.device == train_videos.device
    assert torch.equal(sampled[:, 0], train_videos[:2, 0])
    assert torch.equal(sampled[:, -1], train_videos[:2, -1])

    logits = video_average_logits(frame_model, val_videos[:3].to(DEVICE), 4)
    assert logits.shape == (3, 2) and logits.dtype == torch.float32
    assert logits.device == DEVICE
    assert isinstance(loss_history[-1], float) and loss_history[-1] >= 0.0
    assert isinstance(val_macro_f1, float) and 0.0 <= val_macro_f1 <= 1.0

    assert list(submission.columns) == ['video_id', 'label']
    assert len(submission) == len(test_ids)
    assert submission['video_id'].tolist() == test_ids
    assert submission['label'].dtype == np.dtype('int64')
    assert not submission.isna().any().any()
    print('Day 22 tests passed')


run_day22_tests()

## Day 22 Checklist

- [x] I sampled a fixed number of frames uniformly.
- [x] I tracked `[B,T,C,H,W]`, `[B*T,C,H,W]`, and `[B,T,K]` correctly.
- [x] I trained and evaluated with the same video aggregation rule.
- [x] I computed validation Macro-F1 from integer predictions.
- [x] I preserved test ID order and verified submission columns, dtype, row count, and missing values.
- [x] `run_day22_tests()` prints the required pass message.